# E012-full: complete rebuild with the twin-business features in stage 1 AND stage 2

A **new, separate Kaggle notebook** (own working directory). It never touches the E009/E011 notebook.

| step | what | where / time |
|---|---|---|
| prepare | train + test normalization (France rules, NORM_V2) | CPU, ~20-40 min |
| blocking | E007 union `base,conj:20,bm25:5,bge_native:5,name_noaddr:5` (train recall 0.9818, oracle 0.9942) | **GPU**, several hours |
| dense (optional) | multilingual-e5-small cosines of every candidate pair (native-script names) | GPU, ~1.5 h, skipped automatically when short of session time |
| stage 1 | LightGBM, 3-fold CV, **--feat v6** = v4 (E009) + global name ambiguity + legal-form / business-word / house-number profiles (+ dense) | CPU |
| stage 2 | **--stack-feat v3** on the pruned candidates: profiles, groups, name ambiguity, sibling agreement, candidate-table context, S1-vs-record similarities | CPU |
| selection | per-S1 expected F0.5 with an empty option, one S1 per record, count matching | CPU |

Dev (1% subset): E009 v4 stage 1 0.99033 -> v6 stage 1 0.99296.

**Settings:** Accelerator **GPU T4** · Internet **On** · Persistence **Files** · *Run all*.
Every step is cached, so **Run all resumes** after a stop or a new session.
The notebook stops by itself (red error "STOP: ...") before a step that would not finish inside the 12 h session:
then **Stop session -> start again -> Run all** (Accelerator None is fine once blocking and dense are done).

In [ ]:
# 1. Config
import time
T0 = globals().get("T0") or time.time()   # this session's start (a new session restarts the clock)
EXP       = "20260927-E012-full"
DENSE_WANTED = "e5s"       # "none" skips the dense cosines
STOP_AFTER_GPU = False     # True when GPU quota is < 12 h: stop after blocking/dense, continue with Accelerator None
E007_SPEC = "base,conj:20,bm25:5,bge_native:5,name_noaddr:5"
CV_FRAC   = 0.1
LR        = 0.1
PRUNE_LOSS      = 0.001    # learned blocking before stage 2: may drop 0.1% of the true candidate pairs (lowest p1)
MIN_CAND_RECALL = 0.98     # (0.99 is above the blocker's own recall 0.9818)
DECOY_WEIGHT    = 1.9      # test has ~1.9x train's decoy records (owned by no S1): stage-2 model/rule chosen on a test-like OOF
REPO      = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR  = "/kaggle/working/ber"
WORK      = "/kaggle/working/work"
E009_REF  = {"cv chosen": 0.9647, "India": 0.954, "US": 0.9718, "LB C": 0.950}
def hours():
    return (time.time() - T0) / 3600
def need(h, step):
    if hours() + h > 11.6:
        raise RuntimeError(f"STOP: {step} needs ~{h} h but this session has run {hours():.1f} of 12 h. "
                           "Stop session -> start again -> Run all (everything done so far is reused).")
print(f"session time used: {hours():.2f} h")

In [ ]:
# 2. Dataset, validator, machine
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found"
DATA = os.path.dirname(os.path.dirname(hits[0]))
val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
VALIDATOR = val[0] if val else None
print("DATA =", DATA, "| VALIDATOR =", VALIDATOR)
!mkdir -p {WORK}; du -sh /kaggle/working; free -g; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU"

In [ ]:
# 3. Code, dependencies, tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6
!python -c "import sentence_transformers" 2>/dev/null || pip install -q -U sentence-transformers
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper
import subprocess, json
import pandas as pd
DENSE = "none"   # set by cell 6
def ber(cmd, *extra, dense=None):
    args = ["python", "-m", "ber.run", cmd, "--data", DATA, "--work", WORK, "--exp", EXP, "--cv-frac", str(CV_FRAC),
            "--lr", str(LR), "--df-cap", "2500", "--channels", E007_SPEC, "--feat", "v6",
            "--dense-model", dense or DENSE, "--stack-feat", "v3", *map(str, extra)]
    t = time.time()
    p = subprocess.Popen(args, cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"{cmd} failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- {cmd} done in {(time.time() - t) / 60:.1f} min (session {hours():.1f} h)")
X = lambda name: f"{WORK}/experiments/{EXP}/{name}"
cols = ["macro_f05", "micro_precision", "micro_recall", "f05_singletons", "f05_nonsingletons"]
TAG = "ch-" + E007_SPEC.replace(",", "+").replace(":", "") + "_m0_df2500"

In [ ]:
# 4. Prepare (normalize) train and test once; cached in work/prepared
from ber.normalize import NORM_VERSION
for split in ("train", "test"):
    if not os.path.exists(f"{WORK}/prepared/{split}/s23.parquet"):
        t = time.time()
        subprocess.run(["python", "-c", f"from ber.prepare import load_prepared; load_prepared('{DATA}', '{split}', '{WORK}')"],
                       cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
        print(split, f"prepared in {(time.time() - t) / 60:.1f} min")
    open(f"{WORK}/prepared/{split}/NORM_V{NORM_VERSION}", "w").write("ok")
!ls -la {WORK}/prepared/train {WORK}/prepared/test

In [ ]:
# 5. Blocking (GPU: the bge_native channel is silently skipped without it -> train recall 0.977 instead of 0.982)
import pyarrow.parquet as pq
cand = lambda s: f"{WORK}/{s}/cand_{TAG}.parquet"
todo = [s for s in ("train", "test") if not os.path.exists(cand(s))]
if todo:
    import torch
    assert torch.cuda.is_available(), "set Accelerator = GPU T4: blocking needs it for bge_native"
    import sentence_transformers  # noqa: F401  (bge_native is skipped silently if this is missing)
    for split in todo:
        need(5.0, f"blocking {split}")
        ber("block", "--split", split)
n = {s: pq.ParquetFile(cand(s)).metadata.num_rows for s in ("train", "test")}
print("candidate pairs:", n, "| E007/E008 reference: train 145,218,172, test ~114M")
assert n["train"] > 144_000_000, "too few train candidates: bge_native was skipped (CPU-only blocking gives 141.8M)"

In [ ]:
# 6. Optional dense cosines (GPU). Used only when BOTH splits are done; otherwise the run continues without them.
dense_done = lambda s: bool(glob.glob(f"{WORK}/{s}/dense_{DENSE_WANTED}_full_{TAG}.npy")) and \
                       bool(glob.glob(f"{WORK}/{s}/dense_{DENSE_WANTED}_name_{TAG}.npy"))
if DENSE_WANTED != "none" and not os.path.exists(X("cv_metrics.json")):
    import torch
    for split in ("train", "test"):
        if dense_done(split):
            continue
        if not torch.cuda.is_available():
            print("no GPU in this session: dense skipped"); break
        if hours() > 8.5:
            print(f"session at {hours():.1f} h: dense skipped (stage 1 has priority)"); break
        ber("dense", "--split", split, dense=DENSE_WANTED)
if os.path.exists(X("cv_metrics.json")):   # stage 1 already trained: keep whatever it was trained with
    DENSE = json.load(open(X("cv_metrics.json"))).get("dense_model", DENSE)
else:
    DENSE = DENSE_WANTED if DENSE_WANTED != "none" and all(dense_done(s) for s in ("train", "test")) else "none"
print("DENSE =", DENSE)
if STOP_AFTER_GPU and hours() > 1:
    raise RuntimeError("STOP (STOP_AFTER_GPU): blocking/dense done. Stop session -> Accelerator None -> Run all")

In [ ]:
# 7. Stage-1 CV with v6 (3 folds, selection study, diagnostic slices)
if not os.path.exists(X("cv_metrics.json")):
    need(4.0, "stage-1 CV (features + 3 folds)")
    ber("cv", "--folds", 3, "--no-stress")
    cvj = json.load(open(X("cv_metrics.json"))); cvj["dense_model"] = DENSE
    json.dump(cvj, open(X("cv_metrics.json"), "w"), indent=1)
cv = json.load(open(X("cv_metrics.json")))
print("blocking:", {k: round(v, 4) for k, v in cv["blocking"].items() if k in ("pair_recall", "pairs_per_s1", "oracle_macro_f05")})
print("E009 reference:", E009_REF)
display(pd.DataFrame({k: cv[k] for k in cv if k.startswith("cv") and isinstance(cv[k], dict)}).T[cols].round(4))
print("threshold", cv["threshold"], "| best selection", cv["selection"]["best"], "->", round(cv["selection"]["macro_f05"], 5))
print("profile/ambiguity/dense features", {k: v for k, v in cv["feature_gain_top"].items()
       if k.startswith(("lg_", "w_", "nz_", "hn_", "namelg", "l_name_s1", "r_name_s", "d_"))})
print("top features", list(cv["feature_gain_top"].items())[:12])
display(pd.DataFrame({k[6:]: cv[k] for k in cv if k.startswith("slice_")}).T[cols + ["n_s1"]].round(4))

In [ ]:
# 8. Stage 2 (v3) on the pruned candidates + selection study
if not os.path.exists(X("stack_metrics.json")):
    need(2.0, "stage 2")
    ber("stack", "--folds", 3, "--prune-loss", PRUNE_LOSS, "--min-cand-recall", MIN_CAND_RECALL, "--decoy-weight", DECOY_WEIGHT)
sm = json.load(open(X("stack_metrics.json")))
print("learned blocking:", json.dumps(sm["prune"], indent=1))
rows = [("E009 chosen (LB C 0.950)", E009_REF["cv chosen"]), ("E012 stage 1 best selection", sm["stage1"]["macro_f05"]),
        ("E012 stage 2 best selection", sm["stage2"]["macro_f05"]), ("E012 chosen", sm["chosen"]["macro_f05"])]
display(pd.DataFrame(rows, columns=["variant", "CV macro F0.5"]).round(5))
tl = sm.get("testlike")
if tl:
    print("TEST-LIKE CV (decoys x1.9):", {k: round(tl[k]["macro_f05"], 5) for k in ("stage1", "stage2", "stage2w")}, "| chosen:", tl["chosen_variant"])
print("use_stack:", sm["use_stack"], "| rule:", {k: v for k, v in sm["rule"].items() if k != "cal"})
display(pd.DataFrame({k: sm[k] for k in sm if k.startswith(("chosen", "slice_"))}).T[cols + ["n_s1"]].round(4))
print("top stage-2 features", list(sm["stack_feature_gain_top"].items())[:15])

In [ ]:
# 9. Test prediction and three submissions: A plain, B count-match France only, C count-match every country (E009's best)
import shutil
OUTS = {"A": ("none", "/kaggle/working/output_E012_A"), "B": ("unseen", "/kaggle/working/output_E012_B"),
        "C": ("all", "/kaggle/working/output_E012_C")}
if not os.path.exists(X("test_pairs_proba.parquet")):
    need(5.0, "test prediction (stage 1 on ~114M pairs + stage 2)")
    used = int(subprocess.run(["du", "-sb", "/kaggle/working"], capture_output=True, text=True).stdout.split()[0]) / 2 ** 30
    if used > 13:  # Output quota ~19.5 GiB: train candidates/features are not needed after stage 2 (oof.parquet is kept)
        for f in glob.glob(f"{WORK}/train/*{TAG}*"):
            print("rm", f); os.remove(f)
    !du -sh /kaggle/working
    ber("predict", "--out", OUTS["A"][1])
for k, (cm, od) in OUTS.items():
    if not os.path.exists(f"{od}/matching_results.tsv"):
        ber("select", "--count-match", cm, "--out", od, *(["--cand-from", OUTS["A"][1]] if k != "A" else []))
    cf = f"{od}/candidate_pairs.tsv"
    if os.path.islink(cf):  # a real file for the download / final zip (hard link: no extra disk)
        src = os.path.realpath(cf); os.remove(cf)
        try:
            os.link(src, cf)
        except OSError:
            shutil.copy(src, cf)
    if VALIDATOR:
        !python {VALIDATOR} --matching {od}/matching_results.tsv --candidate {od}/candidate_pairs.tsv --test-dir {DATA}/test --check-ids | tail -2
    info = json.load(open(f"{od}/selection_info.json"))
    print(k, "count_match =", cm, "| delta", info["delta"])
    display(pd.DataFrame(info["per_country"]).T.round(3))
T = pd.read_parquet(X("test_pairs_proba.parquet"), columns=["s1_idx"])
print("test candidate pairs after learned blocking:", len(T), f"({len(T) / 1732544:.1f} per S1)")
!du -sh /kaggle/working

In [ ]:
# 10. Bundle: send /kaggle/working/E012_results.tgz back (metrics + error sample; no submissions inside)
import tarfile
B = "/kaggle/working/E012_results"
os.makedirs(B, exist_ok=True)
for f in ("cv_metrics.json", "stack_metrics.json", "oof_errors.parquet"):
    if os.path.exists(X(f)):
        shutil.copy(X(f), f"{B}/{f}")
for split in ("train", "test"):
    for f in glob.glob(f"{WORK}/{split}/dense_*_info.json"):
        shutil.copy(f, f"{B}/{split}_{os.path.basename(f)}")
for k, (_, od) in OUTS.items():
    if os.path.exists(f"{od}/selection_info.json"):
        shutil.copy(f"{od}/selection_info.json", f"{B}/selection_info_{k}.json")
with tarfile.open("/kaggle/working/E012_results.tgz", "w:gz") as t:
    t.add(B, arcname="E012_results")
print(sorted(os.listdir(B)), os.path.getsize("/kaggle/working/E012_results.tgz") // 1024, "KiB", f"| session {hours():.1f} h")